# Fresh-data WSD decay — break the MRC plateau (5M docs)

Forks the parked **4M** trunk snapshot and anneals LR `1-sqrt -> 0` over the next ~1M
**fresh, unseen** CulturaX chunks (no recycled pool, no repeat). Writes a standalone
`milestone_5000000_decay/`. **Manifest-neutral**: does not touch the trunk manifest, so
nb16 still continues the flat trunk from 4M.

Run on **A100** (the decay is short, ~1.9k steps). Cache build (rebuild fallback only) is GPU-free.
MRC eval is separate (nb17, L4).

In [ ]:
# HF auth (only needed if a 6M rebuild streams CulturaX)
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    import os
    if os.environ.get('HF_TOKEN'):
        login(token=os.environ['HF_TOKEN'])

In [ ]:
%%capture
!pip install -U "huggingface_hub>=1.13.0" transformers accelerate datasets safetensors sentencepiece tokenizers pandas matplotlib tqdm
# NO xformers: NeoBERT's fused SwiGLU NaNs on some torch/CUDA builds; artifacts use a patched pure-torch SwiGLU.
!pip uninstall -y xformers

In [ ]:
import sys, importlib
from pathlib import Path
import torch

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)

PROJECT_ROOT = Path('/content/drive/MyDrive/SALT3') if Path('/content/drive/MyDrive').exists() else Path.cwd() / 'SALT3'
sys.path.insert(0, '/content'); sys.path.insert(0, str(PROJECT_ROOT / 'code'))  # project code FIRST so a stale /content/*.py never shadows the synced modules

import salt3_common as sc; importlib.reload(sc)
import salt3_staged_schedule as sched; importlib.reload(sched)
import salt3_staged_cpt_manifest as mf; importlib.reload(mf)
import salt3_fresh_decay as fd; importlib.reload(fd)
from salt3_common import configure_environment, set_seed, ensure_dir

configure_environment(); set_seed(42)
INIT_ROOT     = PROJECT_ROOT / 'init'
RUNS_ROOT     = ensure_dir(PROJECT_ROOT / 'runs' / 'wsd')
DATASET_CACHE = ensure_dir(PROJECT_ROOT / 'datasets')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
import transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| device', DEVICE)

# Report the masking path. The decay self-handles NeoBERT mask_all even if this salt3_common
# predates the mask_scheme arg, so there is NO hard dependency on the synced salt3_common version:
import inspect
_has_ms = 'mask_scheme' in inspect.signature(sc.make_mlm_collator).parameters
print('salt3_common  ->', sc.__file__, '| has mask_scheme arg:', _has_ms)
print('salt3_schedule->', sched.__file__)
print('decay masking : NeoBERT mask_all', '(via salt3_common)' if _has_ms else '(via built-in fallback)')

## Config
`CEILING_DOCS` defaults to the existing 5M cache (reuse-first). Only bump to `6_000_000` and run
the rebuild cell if the probe reports no unseen room (trunk ran past 4M).

In [ ]:
INIT_NAME         = 'trung_salt_decpertoken_freezealigned'
CEILING_DOCS      = 5_000_000      # reuse existing cache; 6_000_000 only as rebuild fallback
TARGET_TOTAL_DOCS = 5_000_000      # presented budget: 4M flat + ~20% fresh decay
DECAY_FRAC        = 0.20           # ~18% accepted if the cache tail is short (no rebuild)
SHAPE             = '1-sqrt'       # strongest cooldown shape; 'cosine' is the fallback
MAX_SEQ_LEN = 1024; PER_DEVICE_BS = 32; GRAD_ACCUM = 16; PEAK_LR = 1e-4
EVAL_RATIO = 0.02; SEED = 42; MLM_PROB = 0.20
EFF_BATCH = PER_DEVICE_BS * GRAD_ACCUM
arm_dir = RUNS_ROOT / INIT_NAME

## Probe — manifest position & branch decisions (the gate)
Reads the manifest READ-ONLY. Decides: carry optimizer state iff the trunk is parked at 4M;
reuse cache (~18% ok) unless there is no unseen room.

In [ ]:
from transformers import AutoTokenizer
from salt3_common import read_cache_meta, docs_to_chunks

tok = AutoTokenizer.from_pretrained(INIT_ROOT / INIT_NAME / 'model', trust_remote_code=True)
manifest = mf.load_manifest(mf.manifest_path(RUNS_ROOT, INIT_NAME))
assert manifest is not None, 'no manifest for this arm'
chunks_consumed = manifest['chunks_consumed']

meta = read_cache_meta(DATASET_CACHE, tok, CEILING_DOCS, MAX_SEQ_LEN, EVAL_RATIO, SEED)
assert meta is not None, f'no cache_meta for ceiling {CEILING_DOCS:,}; build the ceiling cache first'
ratio, train_chunks = meta['ratio_chunks_per_doc'], meta['train_chunks']

chunks_4M    = docs_to_chunks(4_000_000, ratio)
parked_at_4M = abs(chunks_consumed - chunks_4M) <= EFF_BATCH
unseen       = train_chunks - chunks_consumed
opt_state_path = arm_dir / 'optimizer_state.pt'
base_snapshot  = arm_dir / 'milestone_4000000_stable'
CARRY_OPT = bool(parked_at_4M and opt_state_path.exists())

print(f'chunks_consumed = {chunks_consumed:,}  (4M ~= {chunks_4M:,})')
print(f'parked_at_4M    = {parked_at_4M}')
print(f'unseen chunks   = {unseen:,}  (~{unseen/max(1,EFF_BATCH):.0f} steps)')
print(f'ratio           = {ratio:.3f} chunks/doc')
print(f'carry optimizer = {CARRY_OPT}  (opt-state exists={opt_state_path.exists()})')
print(f'base snapshot   = {base_snapshot.name}  exists={(base_snapshot/"model.safetensors").exists()}')
if unseen < EFF_BATCH:
    print('  WARNING: no unseen data in this cache -> set CEILING_DOCS=6_000_000 and run the rebuild cell')

## (Fallback only) Rebuild 6M cache + prefix-hash check
Run ONLY if the probe warned about no unseen room. GPU-free (~2.5-3h for 6M). The prefix-hash
check guards that the larger cache's first chunks match the old cache, so the 4M snapshot's
continuation is still valid.

In [ ]:
# REBUILD FALLBACK — skip unless the probe said no unseen room.
import hashlib
from salt3_common import make_mlm_datasets
REBUILD_CEILING = 6_000_000

make_mlm_datasets(tokenizer=tok, cache_dir=DATASET_CACHE, num_examples=REBUILD_CEILING,
                  max_seq_len=MAX_SEQ_LEN, eval_ratio=EVAL_RATIO, seed=SEED, chunk_start=0, chunk_end=1)

def _prefix_hash(num_examples, n=2000):
    tr, _ = make_mlm_datasets(tokenizer=tok, cache_dir=DATASET_CACHE, num_examples=num_examples,
                              max_seq_len=MAX_SEQ_LEN, eval_ratio=EVAL_RATIO, seed=SEED,
                              chunk_start=0, chunk_end=n)
    h = hashlib.sha256()
    for row in tr:
        h.update(repr(list(row['input_ids'])).encode())
    return h.hexdigest()

assert _prefix_hash(5_000_000) == _prefix_hash(REBUILD_CEILING), \
    'cache prefix drift -> 4M snapshot continuity broken; do NOT decay on this cache'
print('prefix-hash OK -> set CEILING_DOCS=6_000_000 above and re-run the probe before decaying')

## Run the fresh-data decay (A100)
`run_fresh_decay` asserts contiguity (`chunk_start == chunks_consumed`) and no-repeat
(`window >= steps*eff_batch`), then anneals `1-sqrt -> 0` over the fresh window.

## Pre-flight: prove the decay masking is true NeoBERT mask_all
Builds the exact collator `run_fresh_decay` will use and hard-fails if it isn't 100% [MASK] (no random/keep). Cheap check — run it BEFORE the costly decay.

In [ ]:
# Same collator the decay builds internally (mask_scheme='mask_all', version-independent):
from salt3_common import make_mlm_collator
_decay_collator = sched.build_mlm_collator_compat(make_mlm_collator, tok, MLM_PROB, 'mask_all')
print('mask_all proof:', sched.assert_mask_all(_decay_collator, tok, seq_len=MAX_SEQ_LEN, mlm_probability=MLM_PROB))
# -> {'ok': True, 'masked_fraction': ~0.20, ...}; raises AssertionError if masking is wrong.

In [ ]:
res = fd.run_fresh_decay(
    arm_dir=arm_dir, base_snapshot=base_snapshot, tokenizer=tok, dataset_cache=DATASET_CACHE,
    ceiling_docs=CEILING_DOCS, chunks_consumed=chunks_consumed, max_seq_len=MAX_SEQ_LEN,
    eval_ratio=EVAL_RATIO, seed=SEED, per_device_bs=PER_DEVICE_BS, grad_accum=GRAD_ACCUM,
    target_total_docs=TARGET_TOTAL_DOCS, decay_frac=DECAY_FRAC, peak_lr=PEAK_LR, shape=SHAPE,
    opt_state_path=(opt_state_path if CARRY_OPT else None), rewarmup_steps=300,
    mlm_probability=MLM_PROB, device=DEVICE)
print('decay checkpoint ->', res['out_dir'])
print('window:', res['window'], '| optimizer:', res.get('optimizer'), '| final_lr:', res.get('final_lr'))
print('post-eval:', res.get('post_eval'))

## Stitch the progress log (flat trunk -> decay)
Writes a NEW `metrics_5m_decay.jsonl` = trunk metrics up to the 4M step + the decay per-step
rows tagged `phase:"decay"`. The original `metrics.jsonl` is never modified.

In [ ]:
import json
from salt3_common import read_jsonl, chunks_to_steps, docs_to_chunks

metrics_src = arm_dir / 'metrics.jsonl'
trunk = read_jsonl(metrics_src) if metrics_src.exists() else []
step_4M = chunks_to_steps(docs_to_chunks(4_000_000, ratio), EFF_BATCH)
kept = [{**r, 'phase': 'trunk'} for r in trunk if r.get('step', 0) <= step_4M]
decay_rows = [{'step': step_4M + int(h['step']), 'loss': h['loss'], 'phase': 'decay'}
              for h in res.get('loss_history', [])]

out = arm_dir / 'metrics_5m_decay.jsonl'
with open(out, 'w') as f:
    for r in kept + decay_rows:
        f.write(json.dumps(r) + '\n')
print(f'stitched {len(kept)} trunk + {len(decay_rows)} decay rows -> {out.name} '
      f'(original metrics.jsonl untouched)')

In [ ]:
import matplotlib.pyplot as plt
rows = read_jsonl(out)
pts = [(r['step'], r['loss']) for r in rows if 'loss' in r and 'step' in r]
xs, ys = zip(*pts) if pts else ([], [])
plt.figure(figsize=(8, 4))
plt.plot(xs, ys, lw=1)
plt.axvline(step_4M, ls='--', color='r', label='decay start (4M)')
plt.xlabel('global step'); plt.ylabel('train loss'); plt.legend()
plt.title('Trunk (flat peak LR) -> fresh-data 1-sqrt decay'); plt.show()

In [ ]:
# Free the A100 when done (MRC eval runs separately in nb17 on an L4).
from google.colab import runtime
runtime.unassign()